# Solar Maximum Snapshot Checks

Objective: verify that the v1 snapshot fixture is internally consistent enough for the static web app and classroom walkthrough.

Success criteria:
- Required scalar fields match the grid length.
- Layer labels use the approved disclosure vocabulary.
- Active-region hemispheres and confidence metrics are easy to summarize.


## Setup And Reproducibility

This notebook uses the checked-in fixture generated by `tools/generate_fixture_snapshot.py`. No network calls are required.


In [ ]:
from pathlib import Path
import json
import statistics

def find_repo_root(start):
    for candidate in [start, *start.parents]:
        if (candidate / "apps/web/data/latest-state.json").exists():
            return candidate
    raise FileNotFoundError("apps/web/data/latest-state.json")

REPO_ROOT = find_repo_root(Path.cwd())
SNAPSHOT_PATH = REPO_ROOT / "apps/web/data/latest-state.json"
with SNAPSHOT_PATH.open("r", encoding="utf-8") as handle:
    snapshot = json.load(handle)

CONFIG = {"snapshot": str(SNAPSHOT_PATH), "expected_schema": "solar-state-snapshot.v1"}
print(CONFIG)


## Plan

Hypotheses:
- H1: field arrays are dimensionally consistent with the grid.
- H2: active regions appear in both hemispheres during the maximum fixture.
- H3: disclosure labels are explicit enough for progressive reveal in the UI.


In [ ]:
grid = snapshot["grid"]
expected_len = grid["lon_count"] * grid["lat_count"]
required_fields = ["br_normalized", "continuum_proxy", "confidence"]
checks = {
    field: len(snapshot["fields"][field]["values"]) == expected_len
    for field in required_fields
}
print(checks)
assert all(checks.values())


## Minimal Baseline

Summarize the smallest set of metrics that should remain stable for this fixture unless the model contract changes intentionally.


In [ ]:
br = snapshot["fields"]["br_normalized"]["values"]
confidence = snapshot["fields"]["confidence"]["values"]
regions = snapshot["active_regions"]
metrics = {
    "schema": snapshot["schema_version"],
    "regions": len(regions),
    "north_regions": sum(1 for region in regions if region["lat_deg"] > 0),
    "south_regions": sum(1 for region in regions if region["lat_deg"] < 0),
    "max_abs_br": round(max(abs(value) for value in br), 6),
    "median_confidence": round(statistics.median(confidence), 6),
}
print(metrics)


## Disclosure Vocabulary

The app can reveal advanced details gradually only if every layer has an explicit provenance-style label.


In [ ]:
allowed = {"synthetic", "observed", "blended", "inferred", "degraded"}
labels = {layer["id"]: layer["kind"] for layer in snapshot["layers"]}
invalid = {layer_id: kind for layer_id, kind in labels.items() if kind not in allowed}
print(labels)
assert not invalid


## Results And Notes

Record the printed metrics next to changes that alter the fixture. If a metric changes because the generator changed, update the app fixture and this notebook together.


In [ ]:
result = {
    "field_contract_ok": all(checks.values()),
    "hemispheres_present": metrics["north_regions"] > 0 and metrics["south_regions"] > 0,
    "label_contract_ok": not invalid,
    "operational_use": snapshot["operational_use"],
}
print(result)
assert result["field_contract_ok"]
assert result["hemispheres_present"]
assert result["label_contract_ok"]
assert result["operational_use"] is False


## Next Steps

- Compare Rust-generated snapshots against this fixture contract.
- Add cached live SWPC observations only when provenance and freshness are retained.
- Keep notebook outputs small so diffs remain reviewable.
